# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamzanPUCIT/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

---
From the FlyRank SEO research (March 2026), asked constructively:

1. **Finding #1 — growing content is ~38% longer and ~20% younger.** *Question:* the up/down label comes from a
   30d-vs-prev-30d impression change — are length and age measured **before** that window, and is the comparison
   controlled for client? If big clients write long articles and also grow, length may be standing in for client.
   My own data agrees on age (young pages move more) — but in my slice young pages **drop** more often too, so
   "young" means *volatile*, not only *growing*.
2. **Finding #4 — 365+ day content refreshed within 30 days shows a ~3.2× health boost.** *Question:* the refresh
   and the health score fall in the same 30-day window. Were pages chosen for refresh because they were already
   recovering (selection)? Without a pre-period or a holdout group this is an association, not proof a refresh
   *causes* the boost — the paper's own tags say "directional", which is the right word.

In [1]:
# Setup: load the starter dataset (works in a local clone and in Colab)
from pathlib import Path
import numpy as np
import pandas as pd

RAW_URL = ("https://raw.githubusercontent.com/RamzanPUCIT/flyrank-ml-internship/"
           "main/data/raw/content_refresh_anonymized.csv")
local = [p for p in [Path("../../data/raw/content_refresh_anonymized.csv"),
                     Path("data/raw/content_refresh_anonymized.csv")] if p.exists()]
df = pd.read_csv(local[0] if local else RAW_URL)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("rows x cols:", df.shape)

# My lane slice (same definition in every notebook)
MIN_PREV_IMPR = 100          # page needs real demand in the earlier window
lane = df[df["impressions_prev_30d"] >= MIN_PREV_IMPR].copy()
# proxy target: observed later-window outcome (last 30d vs the 30d before)
lane["demand_drop"] = (lane["impressions_last_30d"] < 0.8 * lane["impressions_prev_30d"]).astype(int)

import json
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
SEED = 42
lane["ctr_prev"] = lane.clicks_prev_30d / lane.impressions_prev_30d * 100
for c in ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]:
    lane["log_" + c] = np.log1p(lane[c])
NUM = ["log_impressions_prev_30d", "log_clicks_prev_30d", "log_sessions_prev_30d", "ctr_prev", "content_age_days",
       "search_volume", "competition", "cpc", "word_count"]
X = pd.concat([lane[NUM].fillna(0), lane[NUM].isna().astype(int).add_suffix("_missing").loc[:, ["search_volume_missing", "word_count_missing"]],
               pd.get_dummies(lane[["content_type", "main_intent"]].fillna("unknown"), dtype=int)], axis=1)
y, groups = lane.demand_drop.values, lane.client_id.values
pct = lambda s: s.rank(pct=True)
lane["baseline_score"] = 0.4 * (1 - pct(lane.content_age_days)) + 0.4 * (1 - pct(lane.ctr_prev)) + 0.2 * pct(lane.log_impressions_prev_30d)
def p_at_k(s, l, k):
    return float(np.asarray(l)[np.argsort(-np.asarray(s), kind="stable")[:k]].mean())
rf = lambda: RandomForestClassifier(200, min_samples_leaf=20, random_state=SEED, n_jobs=-1)

old = lane[lane.content_age_days > 365]
fresh = old.days_since_last_update <= 30
print(f"365+ pages: {len(old):,} | updated in last 30d: {fresh.mean():.0%}")
print("drop rate  updated<=30d:", round(old[fresh].demand_drop.mean(), 3), "| not updated:", round(old[~fresh].demand_drop.mean(), 3),
      "<- same-window comparison, association only")

rows x cols: (30000, 44)


365+ pages: 4,212 | updated in last 30d: 90%
drop rate  updated<=30d: 0.446 | not updated: 0.488 <- same-window comparison, association only


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

---
**Before:** random 5-fold split (pages of one client land in train and test). **After:** GroupKFold by client.
The random split rewards the forest for memorizing client character; the grouped split is the honest number, and
it is the one my capstone reports. Under the honest split the rule baseline stays ahead at precision@50.

In [2]:
rows = []
for name, splitter in [("BEFORE random KFold", KFold(5, shuffle=True, random_state=SEED).split(X)),
                       ("AFTER GroupKFold by client", GroupKFold(5).split(X, y, groups))]:
    auc, p50, b50 = [], [], []
    for tr, te in splitter:
        p = rf().fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
        auc.append(roc_auc_score(y[te], p)); p50.append(p_at_k(p, y[te], 50)); b50.append(p_at_k(lane.baseline_score.values[te], y[te], 50))
    rows.append({"split": name, "rf_auc": np.mean(auc), "rf_precision@50": np.mean(p50), "rule_precision@50": np.mean(b50)})
pd.DataFrame(rows).round(3)

,split,rf_auc,rf_precision@50,rule_precision@50
0,BEFORE random KFold,0.704,0.940,0.820
1,AFTER GroupKFold by client,0.596,0.784,0.808


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

---
Every final feature is from days 31–60 back or a static property. Banned: `trend_*`, all `*_last_30d`, 90-day
totals/averages (overlap the outcome window), `days_since_last_update` (updates inside the outcome window),
tiers/flags. The check below asserts that and shows the "too good" score a leak would give. Failure examples: the
highest-confidence pages that did **not** drop — young, low-CTR, very long pages whose impressions stayed flat.

In [3]:
BANNED = {"trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
          "impressions_90d", "clicks_90d", "ctr", "avg_position", "days_since_last_update"}
assert not BANNED & set(X.columns); print("leak check passed:", X.shape[1], "features")
print("leaky last-30d score precision@50:", p_at_k(-(lane.impressions_last_30d / lane.impressions_prev_30d), y, 50))
oof = np.zeros(len(y))
for tr, te in GroupKFold(5).split(X, y, groups):
    oof[te] = rf().fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
lane["oof"] = oof
lane[lane.demand_drop == 0].nlargest(3, "oof")[["content_age_days", "impressions_prev_30d", "ctr_prev", "word_count",
    "impressions_last_30d", "oof"]].round(3).set_axis(["fail A", "fail B", "fail C"])

leak check passed: 19 features
leaky last-30d score precision@50: 1.0


,content_age_days,impressions_prev_30d,ctr_prev,word_count,impressions_last_30d,oof
fail A,151,1457,0.069,4699.0,1474,0.959
fail B,148,1707,0.059,5781.0,1378,0.946
fail C,151,1753,0.000,5331.0,2277,0.945


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

---
- **Before:** "My model predicts which pages will decline, so refreshing the top pages will save their traffic."
- **After:** "On 18k pages from 30 anonymized sites, a simple three-signal rule *measured* precision@50 ≈ 0.81 on
  client-held-out folds (base rate ≈ 0.63); learned models did not beat it. The queue is *decision-support* for
  where to look first; it does not show that a refresh *causes* recovery."

In [4]:
claims = {"predicts decline": "measured, grouped-CV precision@50 vs base rate",
          "refresh saves traffic": "REMOVED - no experiment, causal claim",
          "model is better": "REMOVED - rule wins at top-50"}
pd.Series(claims, name="status")

predicts decline         measured, grouped-CV precision@50 vs base rate
refresh saves traffic             REMOVED - no experiment, causal claim
model is better                           REMOVED - rule wins at top-50
Name: status, dtype: str

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.